In [12]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

from scipy.spatial.distance import euclidean

In [13]:
df = pd.read_csv("C:/JUPYTERPROJECTS/Crop_Advisory_system/data/Crop_recommendation.csv")
df.head()



,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [14]:
X = df.drop("label", axis=1)
y = df["label"]

In [15]:
# ==============================
# 3. TRAIN-TEST SPLIT (80-20)
# ==============================
train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

[        N    P    K  temperature   humidity        ph    rainfall
 1656   17   16   14    16.396243  92.181519  6.625539  102.944161
 752    37   79   19    27.543848  69.347863  7.143943   69.408782
 892     7   73   25    27.521856  63.132153  7.288057   45.208411
 1041  101   70   48    25.360592  75.031933  6.012697  116.553145
 1179    0   17   30    35.474783  47.972305  6.279134   97.790725
 ...   ...  ...  ...          ...        ...       ...         ...
 1638   10    5    5    21.213070  91.353492  7.817846  112.983436
 1095  108   94   47    27.359116  84.546250  6.387431   90.812505
 1130   11   36   31    27.920633  51.779659  6.475449  100.258567
 1294   11  124  204    13.429886  80.066340  6.361141   71.400430
 860    32   78   22    23.970814  62.355576  7.007038   53.409060
 
 [1760 rows x 7 columns],
         N    P    K  temperature   humidity        ph    rainfall
 1451  101   17   47    29.494014  94.729813  6.185053   26.308209
 1334   98    8   51    26.179346 

In [16]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [17]:
kmeans = KMeans(n_clusters=12, random_state=42)
kmeans.fit(X_train_scaled)

centroids = kmeans.cluster_centers_

In [18]:
# ==============================
# 6. CREATE TRAIN DF (FOR PROPORTIONS)
# ==============================
train_df = X_train.copy()
train_df["label"] = y_train.values
train_df["Cluster"] = kmeans.labels_

In [19]:
# ==============================
# 7. PREDICTION ON TEST DATA
# ==============================
predictions = []

cluster_counts = train_df["Cluster"].value_counts()

for i in range(len(X_test_scaled)):
    
    sample = X_test_scaled[i]
    
    # Step 1: Compute distance to all centroids
    distances = []
    for j, centroid in enumerate(centroids):
        d = euclidean(sample, centroid)
        distances.append((j, d))
    
    # Step 2: Get top 3 nearest clusters
    distances_sorted = sorted(distances, key=lambda x: x[1])
    top_clusters = [c[0] for c in distances_sorted[:3]]
    
    total_samples = cluster_counts[top_clusters].sum()
    
    scores = {}
    
    # Step 3: Apply formula
    for cluster, dist in distances_sorted[:3]:
        
        weight = cluster_counts[cluster] / total_samples
        
        cluster_data = train_df[train_df["Cluster"] == cluster]
        proportions = cluster_data["label"].value_counts(normalize=True)
        
        for crop, prop in proportions.items():
            
            score = (1 / (dist + 1e-6)) * prop * weight
            
            if crop not in scores:
                scores[crop] = 0
            
            scores[crop] += score
    
    # Step 4: Get best prediction
    best_crop = max(scores, key=scores.get)
    
    predictions.append(best_crop)

In [20]:
# ==============================
# 8. RESULT DATAFRAME
# ==============================
result_df = pd.DataFrame({
    "Actual Crop": y_test.values,
    "Predicted Crop": predictions
})

print("\nSample Predictions:")
print(result_df.head())



Sample Predictions:
  Actual Crop Predicted Crop
0   muskmelon         cotton
1  watermelon         cotton
2      papaya         papaya
3      papaya         papaya
4       apple          apple


In [21]:

# ==============================
# 9. ACCURACY
# ==============================
accuracy = accuracy_score(result_df["Actual Crop"], result_df["Predicted Crop"])

print("\nFinal Recommendation Accuracy:", round(accuracy, 4))


Final Recommendation Accuracy: 0.4364
